# 🧪 Colab Test: `train_prefix_qwen_fsdp_wandb.py` with Qwen 2.5 (0.5B)

This notebook runs **unit tests + a tiny training smoke test** for your multi‑GPU trainer, using the lighter **Qwen/Qwen2.5-0.5B-Instruct**.  
You'll see clear **tracebacks** cell-by-cell.

> **Tip:** By default, W&B is set to **disabled** here for quick testing. You can enable it later.

## 0) Runtime check & GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/LLM/Bioreasoner/data/hf/proteinDT")
OUT_DIR  = BASE_DIR / "sft_test_demo"
print(f"Using Google Drive folder as BASE_DIR: {BASE_DIR}")


In [ ]:
%cd /content/drive/MyDrive/ # to the model folder

In [ ]:
!pip -q install "transformers>=4.43.0"
!pip install peft
!pip install accelerate
!pip install bitsandbytes
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 44.6 MB/s eta 0:00:00


In [ ]:
%pip -q install --upgrade pip
%pip -q install --upgrade transformers accelerate datasets sentencepiece wandb safetensors einops
# If you need HF auth for private models:
# from huggingface_hub import notebook_login; notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [ ]:
#import proteinLLM

In [ ]:
import os, sys, types, importlib.util, importlib
from pathlib import Path
from datetime import datetime
import re

In [ ]:
WORKDIR = Path("/testing_place/model") # contains proteinLLM.py, protein_encoder.py, structure_encoder.py, sft_accelerate.py
DATADIR = Path("/testing_place/examples/sft")

In [ ]:
from pathlib import Path

# ==== 🔧 EDIT THESE PATHS ====

SRC_DIR = WORKDIR          # contains proteinLLM.py, protein_encoder.py, structure_encoder.py, sft_accelerate.py
DATA_JSONL = DATADIR / "train_subset_100.jsonl"
VAL_JSONL = DATADIR / "val_subset_100.jsonl"

PROT_SLOT = 1
STRU_SLOT = 3

# Training hyperparams
NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 1
LR = 5e-5
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0
MAX_LENGTH = 1024
MIXED_PRECISION = "bf16"
MAX_SHARD_SIZE = "1GB"
SAVE_EVERY_STEPS = 0
LOG_INTERVAL = 20

# Weights & Biases (optional)
USE_WANDB = True
WANDB_PROJECT = "bigprotein-sft"
WANDB_RUN_NAME = "accelerator_test"
WANDB_ENTITY = ""
WANDB_MODE = "online"

In [ ]:
# one-time
!accelerate config
!export WANDB_API_KEY=

In [ ]:
MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"     # LLM backbone
PROTEIN_CONFIG = "/weights/ProTrek_35M/esm2_t12_35M_UR50D"
STRUCTURE_CONFIG = "/protrek/weights/ProTrek_35M/foldseek_t12_35M"
PROTREK_CKPT    = "/protrek/weights/ProTrek_35M/ProTrek_35M.pt"

BASE_OUTPUT_DIR = Path("/testing_place/ckpt")

def slugify_model_name(name: str) -> str:
    # take the last segment (e.g. "Qwen2.5-0.5B-Instruct") and make it FS-friendly
    leaf = name.split("/")[-1]
    return re.sub(r"[^A-Za-z0-9._-]+", "-", leaf)

In [ ]:
import json
from pathlib import Path

required = {"prompt","response","aa_seq","stru_str"}
assert Path(DATA_JSONL).exists(), f"Training dataset not found: {DATA_JSONL}"

has_pdb = has_cif = has_fasta = total = 0
pdb_keys = ["_af2_pdb_path","pdb_path"]
cif_keys = ["cif_path","mmcif_path"]
fasta_keys = ["fasta_path"]

print("Preview (first 3 examples):")
with open(DATA_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        if i < 3:
            show = {k: (obj.get(k)[:80]+"...") if isinstance(obj.get(k), str) and len(obj.get(k))>80 else obj.get(k)
                    for k in ["prompt","aa_seq","stru_str","response"]}
            print(f"\n--- sample {i} ---\n", show)
        total += 1
        # required keys
        miss = required - set(obj.keys())
        if miss:
            raise KeyError(f"Record {i} missing required keys: {miss}")
        # file presence stats (if paths in the JSONL are absolute)
        for k in pdb_keys:
            p = obj.get(k)
            if isinstance(p, str) and Path(p).exists(): has_pdb += 1
        for k in cif_keys:
            p = obj.get(k)
            if isinstance(p, str) and Path(p).exists(): has_cif += 1
        for k in fasta_keys:
            p = obj.get(k)
            if isinstance(p, str) and Path(p).exists(): has_fasta += 1

print(f"\nTotal records: {total}")
print(f"Existing PDB paths: {has_pdb} | CIF paths: {has_cif} | FASTA paths: {has_fasta}")
if total == 0:
    raise RuntimeError("Empty dataset.")

Preview (first 3 examples):

--- sample 0 ---
 {'prompt': 'You are a professional protein biologist. Based only on the provided inputs, pro...', 'aa_seq': 'MSKGTPSRGKRQTQTHLTCRRCGRMSYHKRHKICSSCGFGRSTRMRSYGWITKRPKVATH', 'stru_str': '<|chain:A|> <|chain_sep|> #ddddvvvvpppddqfdqdppprdraqgpvqragpqqggpndpggdddpvvddd...', 'response': '<thinking>\nThe sequence is relatively short and contains a high proportion of cy...'}

--- sample 1 ---
 {'prompt': 'You are a professional protein biologist. Based only on the provided inputs, pro...', 'aa_seq': 'MSQLCPCGSAVEYSLCCHPYVSGEKVAPDPEHLMRSRYCAFVMQDADYLIKTWHPSCGAAALRAELMAGFAHTEWLGLTV...', 'stru_str': '<|chain:A|> <|chain_sep|> #fdcqqqpprhgcvpalpcclvvvdfdqdpvslvsnlvsclqvvplvsqqvqad...', 'response': '<thinking>\n1. **Signal Peptide and Localization**: The sequence starts with meth...'}

--- sample 2 ---
 {'prompt': 'You are a professional protein biologist. Based only on the provided inputs, pro...', 'aa_seq': 'MSDDHHDDEHFHTGDSGAAATFPKQCSALRKNEHVMIKGRP

In [ ]:
import sys, torch, json
import proteinLLM_train_accelerator as sft_accelerate
from transformers import AutoTokenizer

example = json.loads(open(DATA_JSONL,'r',encoding='utf-8').readline())
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

prompt, response = example["prompt"], example["response"]
p_ids = tok.encode(prompt, add_special_tokens=False)
r_ids = tok.encode(response, add_special_tokens=False)
ids = p_ids + r_ids + [tok.eos_token_id]
la  = [-100]*len(p_ids) + r_ids + [-100]

input_ids = torch.tensor([ids[:256]])
attention_mask = torch.ones_like(input_ids)
labels = torch.tensor([la[:256]])
aa_seq = [example.get("aa_seq")]
stru_str = [example.get("stru_str")]

cfg = sft_accelerate.BigProteinConfig(
    llm_name=MODEL_NAME,
    protein_config=PROTEIN_CONFIG,
    structure_config=STRUCTURE_CONFIG,
    proj_hid=1024, dropout=0.1, train_encoders=False,
    extra_kwargs={"dtype":"auto","single_token_prefix":False,"prefix_len":4},
)
model = sft_accelerate.BigProteinForCausalLM(cfg)

with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels,
                aa_seq=aa_seq, stru_str=stru_str)
print("Smoke forward OK. loss =", float(out.loss))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

No ProTrek checkpoint provided or path not found; encoders stay random-init.
Smoke forward OK. loss = 2.747810125350952


In [ ]:
import sys
from types import SimpleNamespace


TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{slugify_model_name(MODEL_NAME)}-{TIMESTAMP}"

OUTPUT_DIR = BASE_OUTPUT_DIR / RUN_NAME


args = SimpleNamespace(
    dataset=str(DATA_JSONL),
    valset=str(VAL_JSONL) if VAL_JSONL else None,
    output_dir=str(OUTPUT_DIR),
    model_name=str(MODEL_NAME),
    protein_config=str(PROTEIN_CONFIG),
    structure_config=str(STRUCTURE_CONFIG),
    protrek_ckpt=str(PROTREK_CKPT) if PROTREK_CKPT else None,
    prot_slot=int(PROT_SLOT),
    stru_slot=int(STRU_SLOT),
    train_encoders=True,        # set True to also finetune encoders
    num_epochs=int(NUM_EPOCHS),
    batch_size=int(BATCH_SIZE),
    grad_accum=int(GRAD_ACCUM),
    lr=float(LR),
    warmup_ratio=float(WARMUP_RATIO),
    weight_decay=float(WEIGHT_DECAY),
    max_length=int(MAX_LENGTH),
    seed=42,
    clip_grad=1.0,
    mixed_precision=str(MIXED_PRECISION),  # 'bf16' good for A100
    save_every_steps=int(SAVE_EVERY_STEPS),
    max_shard_size=str(MAX_SHARD_SIZE),
    log_interval=int(LOG_INTERVAL),
    # W&B
    wandb=bool(USE_WANDB),
    wandb_project=str(WANDB_PROJECT),
    wandb_run_name=str(WANDB_RUN_NAME) if WANDB_RUN_NAME else None,
    wandb_entity=str(WANDB_ENTITY) if WANDB_ENTITY else None,
    wandb_mode=str(WANDB_MODE) if WANDB_MODE else None,
)

print("Starting training with args:", args)
# If using W&B online, set your key:  import os; os.environ["WANDB_API_KEY"]="..."
sft_accelerate.train(args)

Starting training with args: namespace(dataset='/content/drive/MyDrive/LLM/Bioreasoner/testing_place/examples/sft/train_subset_100.jsonl', valset='/content/drive/MyDrive/LLM/Bioreasoner/testing_place/examples/sft/val_subset_100.jsonl', output_dir='/content/drive/MyDrive/LLM/Bioreasoner/testing_place/ckpt/Qwen2.5-0.5B-Instruct-20251013-064057', model_name='Qwen/Qwen2.5-0.5B-Instruct', protein_config='/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/esm2_t12_35M_UR50D', structure_config='/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/foldseek_t12_35M', protrek_ckpt='/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/ProTrek_35M.pt', prot_slot=1, stru_slot=3, train_encoders=True, num_epochs=1, batch_size=2, grad_accum=1, lr=5e-05, warmup_ratio=0.03, weight_decay=0.0, max_length=1024, seed=42, clip_grad=1.0, mixed_precision='bf16', save_every_steps=0, max_shard_size='1GB', log_interval=20, wandb=True, wandb_project='bigprotein-sft', wan

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zeqi1213 (zeqi1213-brown-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[ProteinEncoder] loaded from slot 1 | missing=3 unexpected=2
[StructureEncoder] loaded from slot 3 | missing=3 unexpected=2
[epoch 1] step 1/10  loss=3.1453  lr=4.878e-05  elapsed=1.7s
[epoch 1] val_loss=2.0706
Saved final checkpoint to /content/drive/MyDrive/LLM/Bioreasoner/testing_place/ckpt/Qwen2.5-0.5B-Instruct-20251013-064057


train/epoch,▁▁▁▁▁▁▁▁▁▁
train/loss,▅▃█▅▄▄▇▁▂▂
train/lr,█▇▇▆▅▃▂▂▁▁
val/epoch,▁
val/loss,▁
train/epoch,1
train/loss,2.20588
train/lr,0
val/epoch,1
val/loss,2.07063


### Check ckpt

In [ ]:
# ===== EDIT THIS =====
OUTPUT_DIR = '/testing_place/ckpt/Qwen2.5-0.5B-Instruct-20251013-064057'

import os, sys, glob, json, torch
from transformers import AutoTokenizer

print(">> Loading model from:", OUTPUT_DIR)
# (You can add low_cpu_mem_usage=False to reduce the 'meta parameter' warning)
model = sft_accelerate.BigProteinForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bp = model.model  # BigProteinQwen

# ensure projector exists and is the expected one
assert hasattr(bp, "prefix_mlp") and isinstance(bp.prefix_mlp, torch.nn.Module), \
    "Expected projector 'prefix_mlp' not found on model.model"

def n_params(m): return sum(p.numel() for p in m.parameters())
def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# 1) Full totals (ground truth)
loaded_total_all = n_params(model)
loaded_train_all = n_trainable(model)

# 2) Component breakdown (explicit, no keyword matching)
parts = {
    "llm"               : getattr(bp, "llm", None),
    "protein_encoder"   : getattr(bp, "protein_encoder", None),
    "structure_encoder" : getattr(bp, "structure_encoder", None),
    "prefix_mlp"        : bp.prefix_mlp,
}

print("\n>> Loaded parameter summary")
print(f"Total parameters (ALL):     {loaded_total_all:,}")
print(f"Trainable parameters (ALL): {loaded_train_all:,}")
for name, mod in parts.items():
    if mod is not None:
        print(f" - {name:18s} params={n_params(mod):,}  trainable={n_trainable(mod):,}")
    else:
        print(f" - {name:18s} (not found)")

# 3) Quick projector input-dim check (should be 1024 for length-fusion)
first_linear = next((m for m in bp.prefix_mlp.modules() if isinstance(m, torch.nn.Linear)), None)
if first_linear:
    print(f"\nProjector (prefix_mlp) first Linear in_features: {first_linear.in_features}  (expect 1024)")
else:
    print("\n(Projector has no Linear submodule found for quick in_features check.)")

# 4) Fresh instance parity check (same config → same total params)
fresh = sft_accelerate.BigProteinForCausalLM(model.config)
fresh_total_all = n_params(fresh)
print(f"\nFresh total params (ALL):  {fresh_total_all:,}")
print(f"Loaded total params (ALL): {loaded_total_all:,}")

# 5) State-dict sanity (keys + shapes should all match)
sd_loaded = model.state_dict()
sd_fresh  = fresh.state_dict()
missing_in_loaded    = sorted(set(sd_fresh.keys()) - set(sd_loaded.keys()))
unexpected_in_loaded = sorted(set(sd_loaded.keys()) - set(sd_fresh.keys()))
shape_mismatches = [(k, tuple(sd_fresh[k].shape), tuple(sd_loaded[k].shape))
                    for k in sd_fresh if k in sd_loaded and sd_fresh[k].shape != sd_loaded[k].shape]

print("\nMissing keys in LOADED (expect 0):   ", len(missing_in_loaded))
print("Unexpected keys in LOADED (expect 0):", len(unexpected_in_loaded))
print("Shape mismatches (expect 0):         ", len(shape_mismatches))

assert not missing_in_loaded, "Missing keys: " + str(missing_in_loaded[:5])
assert not unexpected_in_loaded, "Unexpected keys: " + str(unexpected_in_loaded[:5])
assert not shape_mismatches, "Shape mismatches: " + str(shape_mismatches[:3])

# 6) If a params delta still exists, print it and the projector size
if fresh_total_all != loaded_total_all:
    print(f"\nΔ(fresh - loaded) = {fresh_total_all - loaded_total_all:,}")
    print(f"prefix_mlp params  = {n_params(bp.prefix_mlp):,}")


>> Loading model from: /content/drive/MyDrive/LLM/Bioreasoner/testing_place/ckpt/Qwen2.5-0.5B-Instruct-20251013-064057


Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B-Instruct and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[ProteinEncoder] loaded from slot 1 | missing=3 unexpected=2
[StructureEncoder] loaded from slot 3 | missing=3 unexpected=2


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


>> Loaded parameter summary
Total parameters (ALL):     563,983,229
Trainable parameters (ALL): 563,983,229
 - llm                params=494,032,768  trainable=494,032,768
 - protein_encoder    params=33,992,914  trainable=33,992,914
 - structure_encoder  params=33,989,547  trainable=33,989,547
 - prefix_mlp         params=1,968,000  trainable=1,968,000

Projector (prefix_mlp) first Linear in_features: 1024  (expect 1024)
[ProteinEncoder] loaded from slot 1 | missing=3 unexpected=2
[StructureEncoder] loaded from slot 3 | missing=3 unexpected=2

Fresh total params (ALL):  563,983,229
Loaded total params (ALL): 563,983,229

Missing keys in LOADED (expect 0):    0
Unexpected keys in LOADED (expect 0): 0
Shape mismatches (expect 0):          0
